# Lab 04 · Minería de datos

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 4*

Trabaja sobre tu propia copia del notebook. Todo lo que escribas queda en ella.

In [ ]:
#@title De qué se trata este lab { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">De qué se trata este lab</div><strong>Preguntas que vamos a responder</strong>
<ul>
<li>Cómo se busca un patrón que se repite, cuando nadie te dijo qué patrón buscar</li>
<li>Qué quiere decir que dos cosas se parezcan, y por qué hay más de una respuesta</li>
<li>Cuánto vale una regla del tipo cuando pasa esto, después pasa esto otro</li>
<li>Cómo se mide si el patrón que encontraste es de verdad o es una coincidencia</li>
</ul>
<strong>Al terminar vas a poder</strong>
<ul>
<li>Convertir una bitácora de eventos en algo que un algoritmo pueda buscar</li>
<li>Elegir una medida de distancia sabiendo qué es lo que hace parecidas a dos cosas</li>
<li>Sacar reglas de asociación con apriori y leerlas con soporte, confianza y lift</li>
<li>Distinguir una regla que sirve de una que solo repite lo que ya era frecuente</li>
</ul></div>"""))

In [ ]:
#@title Datos del curso { display-mode: "form" }
#@markdown Corre esta celda. Deja listos los archivos del Observatorio de Datos Energeticos.
import numpy as np, pandas as pd, os, json, sqlite3
if not os.path.exists("centrales.csv"):
    rng = np.random.default_rng(2026)
    centrales = pd.DataFrame([
     ("Central Rio Manso Alto","hidro","Biobio",420,2004),("Central Salto Verde","hidro","Los Lagos",310,1998),
     ("Central Aguas Claras","hidro","Biobio",180,2011),("Central Vega Azul","hidro","Los Lagos",95,2016),
     ("Central Tres Saltos","hidro","Biobio",260,1995),
     ("Parque Solar Pampa Alta","solar","Antofagasta",230,2019),("Parque Solar Llano Seco","solar","Atacama",180,2020),
     ("Parque Solar Sol Naciente","solar","Antofagasta",145,2021),("Parque Solar Quebrada Honda","solar","Atacama",95,2022),
     ("Parque Solar Altiplano","solar","Antofagasta",310,2023),
     ("Eolica Cerro Negro","eolica","Coquimbo",160,2017),("Eolica Punta Ventosa","eolica","Coquimbo",120,2018),
     ("Eolica Loma Fria","eolica","Valparaiso",85,2020),("Eolica Campo Abierto","eolica","Coquimbo",200,2021),
     ("Termoelectrica Bahia Norte","gas","Valparaiso",375,2008),("Termoelectrica Puerto Sur","gas","Biobio",290,2012),
     ("Termoelectrica Valle Central","gas","Metropolitana",210,2006),
     ("Carboelectrica Costa Brava","carbon","Biobio",480,2001),("Carboelectrica Roca Gris","carbon","Antofagasta",350,1999),
     ("Diesel Respaldo Cordillera","diesel","Metropolitana",45,2014),
    ], columns=["central","tecnologia","region","potencia_mw","anio_inicio"])
    fechas = pd.date_range("2024-01-01","2024-12-31",freq="D")
    perfil = np.array([0,0,0,0,0,0,.05,.18,.38,.58,.75,.87,.93,.9,.8,.63,.42,.2,.05,0,0,0,0,0])
    filas=[]
    for _,c in centrales.iterrows():
        p,t = c.potencia_mw, c.tecnologia
        for f in fechas:
            est = 1+0.25*np.cos(2*np.pi*(f.dayofyear-15)/365)
            if t=="solar": base = p*perfil*0.30*est*rng.uniform(.8,1.1)
            elif t=="eolica": base = p*0.36*rng.uniform(.15,1.6,24)
            elif t=="hidro": base = p*0.55*(2-est)*rng.uniform(.9,1.1,24)
            elif t=="gas": base = p*0.68*rng.uniform(.9,1.05,24)
            elif t=="carbon": base = p*0.65*rng.uniform(.95,1.02,24)
            else:
                base = np.zeros(24); base[18:23] = p*0.55*rng.uniform(.8,1,5)
            filas.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                       "central":c.central,"mwh":np.clip(base,0,p).round(2)}))
    centrales.to_csv("centrales.csv", index=False)
    pd.concat(filas, ignore_index=True).to_csv("generacion.csv", index=False)

    # Excel con dos hojas, la segunda con notas en texto libre
    with pd.ExcelWriter("centrales.xlsx") as w:
        centrales.to_excel(w, sheet_name="centrales", index=False)
        pd.DataFrame({"nota":["Potencias declaradas al 31 de diciembre de 2024",
                              "Las centrales de pasada se informan con su potencia maxima"]}
                     ).to_excel(w, sheet_name="notas", index=False)

    # Demanda por región, base de datos SQLite
    regs = ["Antofagasta","Atacama","Coquimbo","Valparaiso","Metropolitana","Biobio","Los Lagos"]
    pobl = [700000,320000,850000,1900000,8100000,1700000,900000]
    perfil_d = np.array([.72,.68,.66,.65,.66,.70,.78,.88,.95,.98,1.0,1.02,1.03,1.0,.97,.96,.97,1.0,1.06,1.10,1.08,.98,.88,.79])
    dem=[]
    for r,p in zip(regs,pobl):
        base_r = p/8000
        for f in fechas:
            inv = 1+0.18*np.cos(2*np.pi*(f.dayofyear-190)/365)
            finde = 0.92 if f.dayofweek>=5 else 1.0
            v = base_r*perfil_d*inv*finde*rng.uniform(.97,1.03,24)
            dem.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                     "region":r,"mwh":v.round(2)}))
    demanda = pd.concat(dem, ignore_index=True)
    demanda.to_csv("demanda.csv", index=False)
    con = sqlite3.connect("demanda.db")
    demanda.to_sql("demanda", con, index=False, if_exists="replace")
    pd.DataFrame({"region":regs,"poblacion":pobl}).to_sql("regiones", con, index=False, if_exists="replace")
    con.close()

    # Precios de nudo de enero, como los entregaría una API REST
    ene = demanda[demanda["fecha"].str.startswith("2024-01")]
    pr = ene.assign(precio_usd_mwh=(40 + ene["mwh"]/ene["mwh"].max()*110
                                    + rng.normal(0,6,len(ene))).clip(40,180).round(2))
    with open("precios_nudo.json","w") as f:
        json.dump({"metadata":{"fuente":"Observatorio de Datos Energeticos",
                               "fecha_consulta":"2024-02-01","unidad":"USD por MWh"},
                   "datos": pr[["fecha","hora","region","precio_usd_mwh"]].to_dict("records")},
                  f)

    # ------------------------------------------------ bitacora de mantenimiento
    # Seiscientos eventos del año, con dos patrones plantados a propósito que el
    # lab va a tener que encontrar y medir.
    rngm = np.random.default_rng(404)
    TIPOS = ["preventivo","correctivo","falla_electrica","falla_mecanica",
             "evento_climatico","inspeccion"]
    PESOS = {"hidro":[.34,.14,.12,.16,.02,.22], "solar":[.38,.12,.16,.08,.02,.24],
             "eolica":[.26,.12,.12,.20,.08,.22], "gas":[.30,.18,.14,.18,.01,.19],
             "carbon":[.28,.18,.14,.20,.01,.19], "diesel":[.14,.52,.10,.12,.01,.11]}
    DURA = {"preventivo":(4,24), "correctivo":(6,72), "falla_electrica":(2,48),
            "falla_mecanica":(8,96), "evento_climatico":(3,36), "inspeccion":(1,8)}
    n_dias = len(fechas)
    invierno = np.isin(fechas.month.to_numpy(), [5,6,7,8])
    peso_clima = np.where(invierno, 4.0, 1.0); peso_clima /= peso_clima.sum()

    ev = []
    for _,c in centrales.iterrows():
        p = np.array(PESOS[c.tecnologia]); p = p/p.sum()
        n = 30 if c.tecnologia=="diesel" else 22
        for tipo, d in zip(rngm.choice(TIPOS, size=n, p=p), rngm.integers(0, n_dias, size=n)):
            ev.append([c.central, int(d), str(tipo)])
        n_cl = 12 if c.tecnologia=="eolica" else 1
        for d in rngm.choice(n_dias, size=n_cl, replace=False, p=peso_clima):
            ev.append([c.central, int(d), "evento_climatico"])

    # Patron 1, el 70 por ciento de las fallas electricas arrastra un correctivo
    # en la misma central dentro de los tres dias siguientes.
    elec = [i for i,e in enumerate(ev) if e[2]=="falla_electrica"]
    for i in sorted(rngm.choice(elec, size=int(round(len(elec)*0.70)), replace=False)):
        ev.append([ev[i][0], min(ev[i][1] + int(rngm.integers(0,4)), n_dias-1), "correctivo"])
    # Patron 2, la mitad de los eventos climaticos trae una falla mecanica el mismo dia.
    clim = [i for i,e in enumerate(ev) if e[2]=="evento_climatico"]
    for i in sorted(rngm.choice(clim, size=int(round(len(clim)*0.50)), replace=False)):
        ev.append([ev[i][0], ev[i][1], "falla_mecanica"])

    nombres = centrales["central"].to_numpy()
    while len(ev) < 600:
        ev.append([str(nombres[int(rngm.integers(0,len(nombres)))]),
                   int(rngm.integers(0,n_dias)),
                   "preventivo" if rngm.random()<0.6 else "inspeccion"])
    ev = ev[:600]

    mant = pd.DataFrame([{"central":c, "fecha":fechas[d].strftime("%Y-%m-%d"),
                          "tipo_evento":t,
                          "duracion_horas":int(rngm.integers(DURA[t][0], DURA[t][1]+1))}
                         for c,d,t in ev])
    mant.sort_values(["fecha","central"], kind="stable").reset_index(drop=True) \
        .to_csv("mantenimiento.csv", index=False)
print("Datos listos, incluida la bitacora de mantenimiento")


## 1. La bitácora y cuatro formas de mirarla

In [ ]:
import pandas as pd

# La bitácora del año, un evento por fila. Esto es lo que llena el turno a mano.
mant = pd.read_csv("mantenimiento.csv", parse_dates=["fecha"])
centrales = pd.read_csv("centrales.csv")

print(mant.shape)
mant.head()

In [ ]:
# El diagnóstico del Módulo 3, ahora sobre una tabla de eventos.
print(mant["tipo_evento"].value_counts())

In [ ]:
# Cuánto dura cada tipo. Una inspección son horas, una falla mecánica son días.
print(mant.groupby("tipo_evento")["duracion_horas"].agg(["count", "mean", "max"]).round(1))

In [ ]:
# Forma 1, conjuntos. Que tipos de evento tuvo cada central en cada mes.
mant["mes"] = mant["fecha"].dt.month
conjuntos = mant.groupby(["central", "mes"])["tipo_evento"].agg(lambda s: sorted(set(s)))

print(len(conjuntos), "conjuntos, uno por central y mes")
print(conjuntos.head(4).to_string())

In [ ]:
# Forma 2, vectores. Cada central es un vector de 24 números, su día promedio.
gen = pd.read_csv("generacion.csv", parse_dates=["fecha"])
perfil = gen.pivot_table(index="central", columns="hora", values="mwh", aggfunc="mean")

print(perfil.shape, "veinte centrales, cada una con 24 horas")
print(perfil.iloc[:3, 10:15].round(1))

In [ ]:
# Forma 3, matriz. Centrales en las filas, tipos de evento en las columnas.
conteos = pd.crosstab(mant["central"], mant["tipo_evento"])

print(conteos.iloc[:4, :4].to_string())

In [ ]:
# Forma 4, secuencias. El orden importa, así que se ordena y se mira el siguiente.
ordenados = mant.sort_values(["central", "fecha"]).copy()
ordenados["siguiente"] = ordenados.groupby("central")["tipo_evento"].shift(-1)

print(ordenados[["central", "fecha", "tipo_evento", "siguiente"]].head(5).to_string(index=False))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Los eventos por tecnología</strong>
<p>Cruza la bitácora con la ficha de cada central y mira qué tipos de evento tiene cada tecnología. Pista, <code>merge</code> por <code>central</code> y después <code>pd.crosstab</code>. Hay una tecnología que se lleva casi todos los eventos climáticos.</p></div>"""))

In [ ]:
# Tu turno
# Cruza mant con centrales y arma la tabla de tecnología contra tipo_evento.
mt = mant.merge(centrales, on="central")

print(pd.crosstab(mt["tecnologia"], mt["tipo_evento"]).head(3).to_string())

## 2. Qué tan parecidas son dos cosas

In [ ]:
from scipy.spatial.distance import euclidean, cosine, jaccard

# Tres parques solares, uno grande, uno mediano y uno chico.
grande = perfil.loc["Parque Solar Altiplano"]
chico = perfil.loc["Parque Solar Quebrada Honda"]

# La euclídea suma las diferencias hora por hora. Mide tamaño.
print("euclídea entre la solar grande y la chica", round(euclidean(grande, chico), 1))

In [ ]:
# El coseno mira solo la forma, no el tamaño. Cero quiere decir misma forma.
print(f"coseno entre la solar grande y la chica   {cosine(grande, chico):.4f}")

In [ ]:
# Y acá viene lo que descoloca. El diésel solo genera de noche.
diesel = perfil.loc["Diesel Respaldo Cordillera"]

print(f"solar grande contra solar chica   euclídea {euclidean(grande, chico):7.1f}"
      f"   coseno {cosine(grande, chico):.4f}")
print(f"solar grande contra el diésel     euclídea {euclidean(grande, diesel):7.1f}"
      f"   coseno {cosine(grande, diesel):.4f}")

In [ ]:
#@title La decisión que nadie te avisa { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">La decisión que nadie te avisa</div><p>Las dos medidas están bien calculadas y dicen cosas opuestas. La euclídea dice que la solar grande y la chica están lejos, porque una genera trescientos megawatt y la otra noventa. El coseno dice que son la misma cosa, porque generan a las mismas horas.</p>
<p>Contra el diésel pasa al revés. El coseno da 0,99, que es casi el máximo, porque uno genera de día y el otro de noche.</p>
<p>No hay una correcta. Hay que decidir si te importa el tamaño o la forma, y esa decisión la tomas tú, no el algoritmo.</p></div>"""))

In [ ]:
from scipy.spatial.distance import pdist, squareform
import numpy as np

# Todas contra todas, de una vez. pdist devuelve la mitad, squareform la cuadra.
distancias = pd.DataFrame(squareform(pdist(perfil.values)),
                          index=perfil.index, columns=perfil.index)
# La diagonal es la distancia de cada central consigo misma y estorba.
distancias = distancias.mask(np.eye(len(distancias), dtype=bool))

print(distancias.iloc[:3, :3].round(1).to_string())

In [ ]:
# Para cada central, cuál es la más parecida. idxmin da el nombre de la columna.
tec = centrales.set_index("central")["tecnologia"]
vecina = distancias.idxmin(axis=1).to_frame("mas_parecida")
vecina["misma_tecnologia"] = tec[vecina.index].values == tec[vecina["mas_parecida"]].values

print(vecina.head(5).to_string())
print("\nacierta la tecnología en", vecina["misma_tecnologia"].sum(), "de", len(vecina))

In [ ]:
# Siete de veinte es apenas mejor que el azar. El problema es la escala.
# Si cada perfil se divide por su máximo, todos quedan entre cero y uno.
normal = perfil.div(perfil.max(axis=1), axis=0)

d2 = pd.DataFrame(squareform(pdist(normal.values)), index=normal.index, columns=normal.index)
d2 = d2.mask(np.eye(len(d2), dtype=bool))
vecina2 = d2.idxmin(axis=1)
aciertos = (tec[vecina2.index].values == tec[vecina2.values].values).sum()

print("con el perfil normalizado acierta en", aciertos, "de", len(vecina2))

In [ ]:
# Jaccard es para conjuntos. Mide qué proporción de los elementos no comparten.
uno = set(conjuntos[("Eolica Campo Abierto", 11)])
otro = set(conjuntos[("Termoelectrica Valle Central", 12)])

print("eólica en noviembre  ", ", ".join(sorted(uno)))
print("térmica en diciembre ", ", ".join(sorted(otro)))
print("comparten", len(uno & otro), "de", len(uno | otro), "tipos, jaccard",
      round(1 - len(uno & otro) / len(uno | otro), 3))

In [ ]:
#@title Ojo con estas distancias { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Ojo con estas distancias</div><p>Entre dos parques solares del curso el coseno da 0,0000 exacto, y entre la hidro y la térmica también. Eso no pasa en un sistema real, es un efecto de cómo se generaron estos datos, con un perfil por tecnología escalado por potencia.</p>
<p>En el Sistema Eléctrico Nacional dos parques solares tienen curvas parecidas pero no idénticas, porque dependen de la nubosidad, de la orientación de los paneles y de las restricciones de la línea que los evacúa. El coseno daría un número chico, no cero.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>La misma matriz, con otra medida</strong>
<p>Vuelve a calcular la matriz de distancias del perfil crudo, pero con <code>metric="cosine"</code> dentro de <code>pdist</code>, y cuenta en cuántas centrales acierta la tecnología. Pista, es un solo argumento y el resto del código no cambia.</p></div>"""))

In [ ]:
# Tu turno
# Agrega metric="cosine" a pdist y vuelve a contar los aciertos.
d3 = pd.DataFrame(squareform(pdist(perfil.values)), index=perfil.index, columns=perfil.index)
d3 = d3.mask(np.eye(len(d3), dtype=bool))
v3 = d3.idxmin(axis=1)

print("aciertos", (tec[v3.index].values == tec[v3.values].values).sum(), "de", len(v3))

## 3. De la bitácora a transacciones

In [ ]:
# Antes de buscar patrones hay que decidir qué es una transacción.
# Acá elegimos central y mes. Todo lo que le pasó a una central en un mes.
transacciones = mant.groupby(["central", "mes"])["tipo_evento"].agg(lambda s: sorted(set(s)))

print(len(transacciones), "transacciones")
print("tipos distintos por transacción, en promedio", round(transacciones.apply(len).mean(), 2))

In [ ]:
# Así se ve una transacción. Es una lista de etiquetas, no números.
print(transacciones.head(6).to_string())

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

# El encoder convierte las listas en una tabla de verdaderos y falsos.
# Una columna por tipo de evento, una fila por transacción.
codificador = TransactionEncoder()
binaria = pd.DataFrame(codificador.fit_transform(transacciones.tolist()),
                       columns=codificador.columns_)

print(binaria.shape)
print(binaria.head(4).to_string())

In [ ]:
# El soporte de un tipo es la proporción de transacciones donde aparece.
print(binaria.mean().sort_values(ascending=False).round(3))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Cambia lo que es una transacción</strong>
<p>Arma las transacciones por central y trimestre en vez de central y mes, y mira cuántas quedan y de qué tamaño. Pista, <code>dt.quarter</code> en vez de <code>dt.month</code>. Piensa antes de correr, el soporte de cada tipo va a subir o a bajar.</p></div>"""))

In [ ]:
# Tu turno
# Cambia mes por trimestre y compara el tamaño promedio de la transacción.
mant["trimestre"] = mant["fecha"].dt.quarter
por_mes = mant.groupby(["central", "mes"])["tipo_evento"].agg(lambda s: sorted(set(s)))

print("por mes      ", len(por_mes), "transacciones, tamaño medio",
      round(por_mes.apply(len).mean(), 2))

In [ ]:
#@title Hasta acá llega la primera clase { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Hasta acá llega la primera clase</div><p>La bitácora quedó convertida en una tabla de verdaderos y falsos, que es la forma que necesitan los algoritmos que vienen. Y de paso quedó decidido qué es una transacción, que es la decisión que más cambia lo que vas a encontrar.</p>
<p>La próxima clase parte buscando qué combinaciones de eventos se repiten más de lo esperable, y termina midiendo si los patrones que encontramos son de verdad.</p>
<p><strong>Cuando vuelvas, ejecuta el notebook desde arriba.</strong> Entorno de ejecución, Ejecutar todo. Colab no guarda el estado entre sesiones.</p></div>"""))

## 4. Lo que se repite

In [ ]:
import warnings
from mlxtend.frequent_patterns import apriori

# Al importarse, mlxtend enciende todos los avisos de deprecación de Python, y
# en Colab eso llena la pantalla con avisos del propio Jupyter. Se vuelve a
# dejar el comportamiento por defecto. Una biblioteca puede cambiar una
# configuración global, y conviene saber mirarlo.
warnings.filterwarnings("ignore", category=DeprecationWarning)

# apriori busca todas las combinaciones que aparecen en al menos el 5 por ciento
# de las transacciones. No hay que programar nada, es una llamada.
frecuentes = apriori(binaria, min_support=0.05, use_colnames=True)
frecuentes["n"] = frecuentes["itemsets"].apply(len)

print(len(frecuentes), "combinaciones frecuentes")
print(frecuentes["n"].value_counts().sort_index().to_string())

In [ ]:
# Las parejas más frecuentes. Estas son las que hay que mirar.
pares = frecuentes[frecuentes["n"] == 2].sort_values("support", ascending=False).copy()

# Un conjunto no tiene orden propio y Python lo recorre distinto en cada
# corrida. Se ordena para que la tabla salga siempre igual.
pares["itemsets"] = pares["itemsets"].apply(lambda s: tuple(sorted(s)))

print(pares.head(6).to_string(index=False))

In [ ]:
from mlxtend.frequent_patterns import fpgrowth

# FP-Growth es otro algoritmo, más rápido en tablas grandes. Da lo mismo.
frecuentes_fp = fpgrowth(binaria, min_support=0.05, use_colnames=True)

print("apriori encuentra ", len(frecuentes))
print("fpgrowth encuentra", len(frecuentes_fp))

In [ ]:
# El umbral es la decisión del bloque. Bajarlo encuentra más y casi todo ruido.
for umbral in (0.03, 0.05, 0.10, 0.20):
    print(umbral, "->", len(apriori(binaria, min_support=umbral, use_colnames=True)),
          "combinaciones")

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Busca los tríos</strong>
<p>Con <code>min_support=0.05</code>, muestra las combinaciones de tres eventos ordenadas por soporte. Pista, ya está la columna <code>n</code>. Mira si los tríos que aparecen son los pares frecuentes con un evento común agregado.</p></div>"""))

In [ ]:
# Tu turno
# Cambia el 2 por un 3 para ver los trios en vez de las parejas.
trios = frecuentes[frecuentes["n"] == 2].sort_values("support", ascending=False).copy()
trios["itemsets"] = trios["itemsets"].apply(lambda s: tuple(sorted(s)))

print(trios.head(4).to_string(index=False))

## 5. Reglas de asociación

In [ ]:
from mlxtend.frequent_patterns import association_rules

# Una combinación frecuente no es una regla. La regla tiene dirección,
# si pasa esto entonces pasa esto otro.
reglas = association_rules(frecuentes, metric="confidence", min_threshold=0.5)
reglas["si"] = reglas["antecedents"].apply(lambda s: ", ".join(sorted(s)))
reglas["entonces"] = reglas["consequents"].apply(lambda s: ", ".join(sorted(s)))
COLS = ["si", "entonces", "support", "confidence", "lift"]

print(len(reglas), "reglas con confianza de al menos 0,5")

In [ ]:
# Las reglas simples, de un evento a un evento, ordenadas por lift.
simples = reglas[(reglas["antecedents"].apply(len) == 1)
                 & (reglas["consequents"].apply(len) == 1)]

print(simples.sort_values("lift", ascending=False)[COLS].head(6).round(3).to_string(index=False))

In [ ]:
#@title Los tres números de una regla { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Los tres números de una regla</div><p><strong>Soporte.</strong> En qué proporción de todas las transacciones aparece la combinación completa. Dice si la regla se aplica seguido o casi nunca.</p>
<p><strong>Confianza.</strong> De las transacciones donde pasó el primer evento, en cuántas pasó también el segundo. Es la que suena a promesa, y es la que engaña.</p>
<p><strong>Lift.</strong> Cuántas veces más probable es el segundo evento cuando pasó el primero, comparado con lo que pasaría al azar. Uno quiere decir que el primero no aporta nada. Es el número que hay que mirar.</p></div>"""))

In [ ]:
# La trampa de la confianza. Estas dos reglas tienen confianza parecida.
mirar = simples[simples["entonces"].isin(["correctivo", "preventivo"])]

print(mirar.sort_values("confidence", ascending=False)[COLS].head(6).round(3).to_string(index=False))

In [ ]:
# Reglas con más de un evento a la izquierda, que suelen tener el lift más alto.
compuestas = reglas[reglas["antecedents"].apply(len) > 1]

print(compuestas.sort_values("lift", ascending=False)[COLS].head(5).round(3).to_string(index=False))

In [ ]:
#@title Esto no dice que una cosa cause la otra { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Esto no dice que una cosa cause la otra</div><p>La regla dice que cuando en un mes hubo una falla eléctrica, en ese mismo mes hubo un correctivo el 83 por ciento de las veces. Eso es una coincidencia medida, no una causa.</p>
<p>Acá sabemos que sí hay causa, porque nosotros plantamos el patrón. En un archivo de verdad no lo sabrías, y el mismo número podría salir porque las dos cosas dependen de un tercero, como que la central es vieja, o simplemente porque hay pocos casos.</p>
<p>La regla sirve para decidir dónde mirar, no para cerrar el caso.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Las reglas que terminan en correctivo</strong>
<p>Filtra las reglas cuyo <code>entonces</code> sea <code>correctivo</code> y ordénalas por lift. Son las que le sirven al que programa las cuadrillas, porque le dicen qué eventos anuncian trabajo.</p></div>"""))

In [ ]:
# Tu turno
# Filtra por entonces igual a correctivo y ordena por lift.
print(reglas.sort_values("lift", ascending=False)[COLS].head(4).round(3).to_string(index=False))

## 6. Patrones en secuencias

In [ ]:
# Las reglas del bloque anterior no saben de orden. Dentro de un mes, da igual
# que el correctivo haya sido antes o después de la falla. Acá sí importa.
ordenados["dias_hasta_siguiente"] = (ordenados.groupby("central")["fecha"].shift(-1)
                                     - ordenados["fecha"]).dt.days

print(ordenados[["central", "fecha", "tipo_evento", "siguiente",
                 "dias_hasta_siguiente"]].head(6).to_string(index=False))

In [ ]:
# La matriz de transición. Cada fila suma uno, y dice qué viene después.
transicion = pd.crosstab(ordenados["tipo_evento"], ordenados["siguiente"],
                         normalize="index")

print(transicion.round(3).to_string())

In [ ]:
# La fila que importa. Después de una falla eléctrica, casi siempre un correctivo.
print(transicion.loc["falla_electrica"].sort_values(ascending=False).round(3).to_string())

In [ ]:
# Y cuánto se demora. La mediana de días entre la falla y lo que viene después.
electricas = ordenados[ordenados["tipo_evento"] == "falla_electrica"]

print(electricas["dias_hasta_siguiente"].describe().round(1).to_string())

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>La otra fila</strong>
<p>Mira la fila de <code>evento_climatico</code> en la matriz de transición y compárala con la de <code>falla_electrica</code>. Las dos tienen un patrón plantado, pero uno se ve mucho más fuerte que el otro. Piensa por qué antes de mirar el bloque 7.</p></div>"""))

In [ ]:
# Tu turno
# Cambia falla_electrica por evento_climatico.
print(transicion.loc["falla_electrica"].sort_values(ascending=False).round(3).head(3).to_string())

## 7. Contra la verdad

In [ ]:
#@title Lo que este bloque no se puede hacer en la vida real { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Lo que este bloque no se puede hacer en la vida real</div><p>Igual que en el Módulo 3, acá tenemos una ventaja que el trabajo no da. Sabemos exactamente qué patrones se plantaron en la bitácora, porque los pusimos nosotros al generarla.</p>
<p>Son dos. El 70 por ciento de las fallas eléctricas arrastra un correctivo en la misma central dentro de los tres días siguientes. Y la mitad de los eventos climáticos trae una falla mecánica el mismo día.</p>
<p>Ahora vamos a ver si los números que sacamos se parecen a eso.</p></div>"""))

In [ ]:
# Patrón 1, medido directo. Para cada falla eléctrica, hubo un correctivo
# en la misma central dentro de los tres días siguientes.
electricas = mant[mant["tipo_evento"] == "falla_electrica"]
correctivos = mant[mant["tipo_evento"] == "correctivo"]

con_correctivo = 0
for fila in electricas.itertuples():
    cerca = correctivos[(correctivos["central"] == fila.central)
                        & (correctivos["fecha"] >= fila.fecha)
                        & (correctivos["fecha"] <= fila.fecha + pd.Timedelta(days=3))]
    con_correctivo += len(cerca) > 0

print(len(electricas), "fallas eléctricas")
print(con_correctivo, "con correctivo dentro de tres días =",
      round(100 * con_correctivo / len(electricas), 1), "por ciento")

In [ ]:
# Patrón 2, la mitad de los climáticos con una falla mecánica el mismo día.
climaticos = mant[mant["tipo_evento"] == "evento_climatico"]
mecanicas = mant[mant["tipo_evento"] == "falla_mecanica"]

con_mecanica = 0
for fila in climaticos.itertuples():
    mismo = mecanicas[(mecanicas["central"] == fila.central)
                      & (mecanicas["fecha"] == fila.fecha)]
    con_mecanica += len(mismo) > 0

print(len(climaticos), "eventos climáticos")
print(con_mecanica, "con falla mecánica el mismo día =",
      round(100 * con_mecanica / len(climaticos), 1), "por ciento")

In [ ]:
# Los cuatro números del mismo patrón, uno al lado del otro.
regla = simples[(simples["si"] == "falla_electrica")
                & (simples["entonces"] == "correctivo")]

print(f"lo que plantamos al generar el archivo  {70.0:5.1f} por ciento")
print(f"medido, con ventana de tres días        "
      f"{100 * con_correctivo / len(electricas):5.1f} por ciento")
print(f"la confianza de la regla, el mismo mes  "
      f"{100 * float(regla['confidence'].iloc[0]):5.1f} por ciento")
print(f"la transición, el evento siguiente      "
      f"{100 * transicion.loc['falla_electrica', 'correctivo']:5.1f} por ciento")

In [ ]:
#@title Tres números distintos, y los tres correctos { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tres números distintos, y los tres correctos</div><p>El patrón plantado es uno solo y las tres mediciones dan distinto, porque cada una responde una pregunta distinta.</p>
<p>La medición directa pregunta si hubo un correctivo dentro de tres días. La confianza de la regla pregunta si hubo un correctivo en el mismo mes, que es una ventana más ancha y por eso da más alto. La matriz de transición pregunta si el correctivo fue el evento inmediatamente siguiente, sin límite de días.</p>
<p>Cuando alguien te muestre un porcentaje sacado de reglas de asociación, la primera pregunta es cuál era la ventana. Sin eso el número no quiere decir nada.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Mueve la ventana</strong>
<p>Repite la medición del patrón 1 con una ventana de un día y con una de siete, y anota los tres porcentajes. Es la forma más rápida de ver cuánto depende el resultado de una decisión que casi nadie escribe.</p></div>"""))

In [ ]:
# Tu turno
# Cambia el 3 por un 1 y después por un 7, y anota los tres porcentajes.
ventana = 3
cuenta = 0
for fila in electricas.itertuples():
    cerca = correctivos[(correctivos["central"] == fila.central)
                        & (correctivos["fecha"] >= fila.fecha)
                        & (correctivos["fecha"] <= fila.fecha + pd.Timedelta(days=ventana))]
    cuenta += len(cerca) > 0

print("ventana de", ventana, "días ->", round(100 * cuenta / len(electricas), 1), "por ciento")

In [ ]:
#@title Puntos clave { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Puntos clave</div><ul>
<li>Los mismos datos se pueden mirar como conjuntos, como vectores, como matriz o como secuencia, y cada forma habilita otras preguntas</li>
<li>Parecerse no tiene una sola definición. La euclídea mide tamaño, el coseno mide forma y Jaccard mide cuánto comparten dos conjuntos</li>
<li>Sobre datos con escalas distintas, normalizar antes de medir distancias cambia el resultado por completo</li>
<li>Decidir qué es una transacción es la decisión que más influye en lo que vas a encontrar</li>
<li>De los tres números de una regla, el lift es el único que dice si el antecedente aporta algo</li>
<li>Una regla no prueba una causa, y el porcentaje que promete depende de una ventana que casi nunca viene escrita</li>
</ul>
<p>En el Lab 05 vamos a agrupar sin decirle al algoritmo qué grupos buscar, que es el paso siguiente de esto mismo.</p></div>"""))